# 《Let's build GPT》跟敲笔记（进行时）

> 本文是跟随 Karpathy《Let's build GPT: from scratch, in code, spelled out》视频的逐段跟敲记录。
> **当前进度：单头自注意力。** 后面还有多头注意力、FFN、Block 堆叠、完整训练——那些在 `v2.py`（同文件夹）里全部补齐，两份文件对照着看。

## 路线图

```
字符级分词 → 滑动窗口数据集 → Bigram（查表） → 训练循环
  → 玩具版注意力（累积平均） → 自注意力 Head（Q/K/V）
  → [下一步] 缩放 + 多头 + FFN + Block → 完整 GPT（见 v2.py）
```

每一节前面的 Markdown 单元解释「这段代码在干什么、为什么」；代码里只加必要的行内注释。

## 1. 数据加载：一整本莎士比亚

`input.txt` 是 tiny shakespeare 数据集（约 1.1M 字符的莎士比亚剧本合集）。语言模型的第一步永远是：**把文本读成一个长字符串**。

In [2]:
# 读取 tinyshakespeare 数据集（约 1.1M 字符的莎士比亚剧本合集）
# read it in to inspect it
with open('input.txt', 'r', encoding='utf-8') as f:
    text = f.read()

In [3]:

print("length of dataset in characters: ", len(text))

length of dataset in characters:  1115394


In [4]:

# let's look at the first 1000 characters
print(text[:1000])

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us kill him, and we'll have corn at our own price.
Is't a verdict?

All:
No more talking on't; let it be done: away, away!

Second Citizen:
One word, good citizens.

First Citizen:
We are accounted poor citizens, the patricians good.
What authority surfeits on would relieve us: if they
would yield us but the superfluity, while it were
wholesome, we might guess they relieved us humanely;
but they think we are too dear: the leanness that
afflicts us, the object of our misery, is as an
inventory to particularise their abundance; our
sufferance is a gain to them Let us revenge this with
our pikes, ere we become rakes: for the gods know I
speak this in hunger for bread, not in thirst for revenge.



## 2. 字符级分词：把文字变成数字

模型只认数字，所以先建一套「字符 ↔ 整数」的字典：

- `chars`：数据集中出现过的所有字符（这里 65 个，含换行、标点、大小写字母）；
- `stoi` / `itos`：字符→编号 / 编号→字符 两张映射表；
- `encode("hii there")` → `[46, 47, 47, 1, 58, 46, 43, 56, 43]`，`decode` 再变回字符串。

> 这是最原始的 tokenizer。GPT-2 用的是 byte-level BPE（5 万词表），但原理同款：一段文本 ↔ 一串整数。

In [5]:
# here are all the unique characters that occur in this text
chars = sorted(list(set(text)))   # 数据集中出现过的所有字符，排序保证跨运行顺序一致
vocab_size = len(chars)           # 词表大小 = 65（换行/标点/大小写字母）
print(''.join(chars))
print(vocab_size)


 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz
65


In [6]:
# create a mapping from characters to integers
stoi = { ch:i for i,ch in enumerate(chars) }  # 字符 → 编号
itos = { i:ch for i,ch in enumerate(chars) }  # 编号 → 字符
encode = lambda s: [stoi[c] for c in s] # encoder: take a string, output a list of integers
decode = lambda l: ''.join([itos[i] for i in l]) # decoder: take a list of integers, output a string

print(encode("hii there"))
print(decode(encode("hii there")))

[46, 47, 47, 1, 58, 46, 43, 56, 43]
hii there


## 3. 整本编码 + 90/10 切分

`data = torch.tensor(encode(text))` 把 111 万个编号装进一个长张量；前 90% 当训练集，后 10% 当验证集。验证集只用来评估，不参与训练。

In [7]:
# let's now encode the entire text dataset and store it into a torch.Tensor
import torch # we use PyTorch: https://pytorch.org
data = torch.tensor(encode(text), dtype=torch.long)
print(data.shape, data.dtype)
print(data[:1000]) # the 1000 characters we looked at earier will to the GPT look like this

torch.Size([1115394]) torch.int64
tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 14, 43, 44,
        53, 56, 43,  1, 61, 43,  1, 54, 56, 53, 41, 43, 43, 42,  1, 39, 52, 63,
         1, 44, 59, 56, 58, 46, 43, 56,  6,  1, 46, 43, 39, 56,  1, 51, 43,  1,
        57, 54, 43, 39, 49,  8,  0,  0, 13, 50, 50, 10,  0, 31, 54, 43, 39, 49,
         6,  1, 57, 54, 43, 39, 49,  8,  0,  0, 18, 47, 56, 57, 58,  1, 15, 47,
        58, 47, 64, 43, 52, 10,  0, 37, 53, 59,  1, 39, 56, 43,  1, 39, 50, 50,
         1, 56, 43, 57, 53, 50, 60, 43, 42,  1, 56, 39, 58, 46, 43, 56,  1, 58,
        53,  1, 42, 47, 43,  1, 58, 46, 39, 52,  1, 58, 53,  1, 44, 39, 51, 47,
        57, 46, 12,  0,  0, 13, 50, 50, 10,  0, 30, 43, 57, 53, 50, 60, 43, 42,
         8,  1, 56, 43, 57, 53, 50, 60, 43, 42,  8,  0,  0, 18, 47, 56, 57, 58,
         1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 18, 47, 56, 57, 58,  6,  1, 63,
        53, 59,  1, 49, 52, 53, 61,  1, 15, 39, 47, 59, 57,  1, 25, 39, 56, 41,
      

In [8]:
# Let's now split up the data into train and validation sets
n = int(0.9*len(data)) # first 90% will be train, rest val
train_data = data[:n]
val_data = data[n:]
# 验证集只用来体检过拟合，永远不参与训练

## 4. 语言模型的训练信号：滑动窗口

`block_size = 8` 意味着一次最多看 8 个字符。关键在 x 和 y 是**错开一位**的：

```
x = [18, 47, 56, 57, 58, 1, 46, 43]   ← 输入
y = [47, 56, 57, 58, 1, 46, 43, 39]   ← 目标（每个位置的下一个字符）
```

一个长度为 T 的窗口里藏着 **T 个训练样本**：「看到第 1 个字符猜第 2 个」、「看到前 2 个猜第 3 个」……`get_batch` 就是随机抽 4 个这样的窗口（batch_size=4），堆成 `(B, T)` 的输入和目标。

In [9]:
block_size = 8
x = train_data[:block_size]
y = train_data[1:block_size+1]   # y 是 x 右移一位：每个位置的标签 = 下一个字符
# 逐个打印：看到前 t+1 个字符 → 预测下一个字符
for t in range(block_size):
    context = x[:t+1]
    target = y[t]
    print(f"when input is {context} the target: {target}")

when input is tensor([18]) the target: 47
when input is tensor([18, 47]) the target: 56
when input is tensor([18, 47, 56]) the target: 57
when input is tensor([18, 47, 56, 57]) the target: 58
when input is tensor([18, 47, 56, 57, 58]) the target: 1
when input is tensor([18, 47, 56, 57, 58,  1]) the target: 15
when input is tensor([18, 47, 56, 57, 58,  1, 15]) the target: 47
when input is tensor([18, 47, 56, 57, 58,  1, 15, 47]) the target: 58


In [10]:
torch.manual_seed(1337)
batch_size = 4 # how many independent sequences will we process in parallel?
block_size = 8 # what is the maximum context length for predictions?

def get_batch(split):
    # generate a small batch of data of inputs x and targets y
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))  # 随机抽 batch_size 个窗口起点
    x = torch.stack([data[i:i+block_size] for i in ix])         # (B, T) 输入
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])     # (B, T) 目标（右移一位）
    return x, y

xb, yb = get_batch('train')
print('inputs:')
print(xb.shape)
print(xb)
print('targets:')
print(yb.shape)
print(yb)

print('----')

for b in range(batch_size): # batch dimension
    for t in range(block_size): # time dimension
        context = xb[b, :t+1]
        target = yb[b,t]
        print(f"when input is {context.tolist()} the target: {target}")

inputs:
torch.Size([4, 8])
tensor([[24, 43, 58,  5, 57,  1, 46, 43],
        [44, 53, 56,  1, 58, 46, 39, 58],
        [52, 58,  1, 58, 46, 39, 58,  1],
        [25, 17, 27, 10,  0, 21,  1, 54]])
targets:
torch.Size([4, 8])
tensor([[43, 58,  5, 57,  1, 46, 43, 39],
        [53, 56,  1, 58, 46, 39, 58,  1],
        [58,  1, 58, 46, 39, 58,  1, 46],
        [17, 27, 10,  0, 21,  1, 54, 39]])
----
when input is [24] the target: 43
when input is [24, 43] the target: 58
when input is [24, 43, 58] the target: 5
when input is [24, 43, 58, 5] the target: 57
when input is [24, 43, 58, 5, 57] the target: 1
when input is [24, 43, 58, 5, 57, 1] the target: 46
when input is [24, 43, 58, 5, 57, 1, 46] the target: 43
when input is [24, 43, 58, 5, 57, 1, 46, 43] the target: 39
when input is [44] the target: 53
when input is [44, 53] the target: 56
when input is [44, 53, 56] the target: 1
when input is [44, 53, 56, 1] the target: 58
when input is [44, 53, 56, 1, 58] the target: 46
when input is [44, 53

## 5. 第一个模型：Bigram（查表）

最简语言模型：**一个 65×65 的嵌入表，第 i 行就是「当前字符 i 的下一个字符分数」**——没有隐藏层、没有注意力，纯查表。

- 输入 `(B,T)` → 查表 → `logits (B,T,65)`；
- 交叉熵要求把 logits 拍平成 `(B×T, 65)`、目标拍平成 `(B×T)`；
- `generate`：取最后一个位置的 logits → softmax → 按概率采样一个字符 → 拼回序列，循环。

初始 loss ≈ 4.88（理论随机值 ln 65 ≈ 4.17，初始化还没校准），生成的是纯乱码——这就是后面所有改进的**基线**。

In [11]:
import torch
import torch.nn as nn
from torch.nn import functional as F
torch.manual_seed(1337)

class BigramLanguageModel(nn.Module):

    def __init__(self, vocab_size):
        super().__init__()
        # each token directly reads off the logits for the next token from a lookup table
        # 65×65 查表：第 i 行 = 当前字符是 i 时，下一个字符的原始分数（logits）
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, idx, targets=None):

        # idx and targets are both (B,T) tensor of integers
        logits = self.token_embedding_table(idx) # (B,T,C) —— 查表即预测
        if targets is None:
            loss=None
        else:
            B, T, C =  logits.shape
            logits = logits.view(B*T,C)   # cross_entropy 要求 (样本数, 类别数)
            targets = targets.view(B*T)   # 目标拍平成一维

            loss = F.cross_entropy(logits, targets)
        return logits, loss

    def generate(self, idx, max_new_tokens):
        # idx is (B, T) array of indices in the current context
        for _ in range(max_new_tokens):
            # get the predictions
            logits, loss = self(idx)
            # focus only on the last time step
            logits = logits[:, -1, :] # becomes (B, C) —— 最后一个位置的「答案」就是下一个字符
            # apply softmax to get probabilities
            probs = F.softmax(logits, dim=-1) # (B, C)
            # sample from the distribution
            idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)
            # append sampled index to the running sequence
            idx = torch.cat((idx, idx_next), dim=1) # (B, T+1)
        return idx

m = BigramLanguageModel(vocab_size)
out , loss= m(xb, yb)
print(out.shape)
print(loss)

print(decode(m.generate(torch.zeros((1, 1), dtype=torch.long), max_new_tokens=100)[0].tolist()))

torch.Size([32, 65])
tensor(4.8786, grad_fn=<NllLossBackward0>)

SKIcLT;AcELMoTbvZv C?nq-QE33:CJqkOKH-q;:la!oiywkHjgChzbQ?u!3bLIgwevmyFJGUGp
wnYWmnxKWWev-tDqXErVKLgJ


## 6. 训练循环五步曲

```
取 batch → 前向算 loss → optimizer.zero_grad() → loss.backward() → optimizer.step()
```

AdamW + lr=1e-3，训 1000 步后 loss 从 4.88 降到 **3.70**。生成结果从纯乱码变成「有字母统计规律的乱码」——bigram 只学到了相邻两个字符的统计。

In [12]:
# create a PyTorch optimizer
optimizer = torch.optim.AdamW(m.parameters(), lr=1e-3)


In [13]:
batch_size = 32
# 训练五步曲：取 batch → 前向 → zero_grad → backward → step
for steps in range(1000):
    # sample a batch of data
    xb, yb = get_batch('train')

    # evaluate the loss
    logits, loss = m(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

print(loss.item())

3.704136848449707


In [14]:
print(decode(m.generate(idx = torch.zeros((1, 1), dtype=torch.long), max_new_tokens=300)[0].tolist()))


Wh;;Sq.f ustNzknc
kwgOj$dhPWr,SV?hsusiKpgXXUh;Apmem d?hESXI.i;TrJgkiF-oKbXCAA -botrngFCHAUQkn$

pn$w-gHoi?wtd!
LLULIfSK'bAw :M.ZtOptXEQcL?hfaofqbPd?OnonQQJMap$aypupIBYGUsZaI'ottllo..k$W$Akp?yl?ajKlzY!lx&QQLW? t,bXFkyhl-dmVsHeckhRl,jSClgjuk:3Iv
?OqlrV;!Plxfzgy;;
'mRjuBQ&xk!$
h
SiruDJgKuDny,S$ERf.?GSV


## 7. 注意力前传：从「累积平均」看懂掩码矩阵

注意力很难直接啃，视频先用一个玩具例子搭直觉：让每个位置的输出 = **它前面所有位置的加权平均**，且不许看未来。

三个版本做同一件事，逐步接近真正的注意力：

1. **版本 1（双层循环）**：老实巴交地 `mean(x[:t+1])`；
2. **版本 2（下三角矩阵）**：`tril` 矩阵归一化后 `wei @ x`，一次矩阵乘法完成全部位置的累积平均——**矩阵乘法就是并行版的循环**；
3. **版本 3（softmax）**：把掩码处填 `-inf` 再 softmax——权重从「固定平均」变成「可被训练调整」。这一步凑齐了注意力的两个核心零件：**因果掩码 + softmax**。

```
wei = torch.zeros((T,T))
wei = wei.masked_fill(tril == 0, float('-inf'))   # 未来位置 → -inf
wei = F.softmax(wei, dim=-1)                       # 每行归一化成权重
```

三个版本互相 `allclose` 验证相等——这是视频里「用已知正确的笨办法验证聪明办法」的典型套路。

In [15]:
# 玩具例子：4 个样本、8 个时间步、2 个特征通道
# consider the following toy example:
torch.manual_seed(1337)
B,T,C = 4,8,2 # batch, time, channels
x = torch.randn(B,T,C)
x.shape

torch.Size([4, 8, 2])

In [16]:
# 版本 1（朴素循环）：x[b,t] = 前 t+1 个位置的平均值（只看过去）
# We want x[b,t] = mean_{i<=t} x[b,i]
xbow = torch.zeros((B,T,C))
for b in range(B):
    for t in range(T):
        xprev = x[b,:t+1] # (t,C)
        xbow[b,t] = torch.mean(xprev, 0)

In [17]:
# 版本 2（矩阵版）：tril 下三角 + 行归一化，一次矩阵乘法完成同样的累积平均
wei = torch.tril(torch.ones(T, T))
wei = wei / wei.sum(1, keepdim=True)
xbow2 = wei @ x # (B,T,T) @ (B,T,C) ------>  (B,T,C)
torch.allclose(xbow,xbow2,atol=1e-6)

True

In [18]:
# 版本 3（softmax 版）：掩码处填 -inf → softmax 归一化 → 与上面完全等价（allclose=True）
# version 3: use Softmax
tril = torch.tril(torch.ones(T, T))
wei = torch.zeros((T,T))
wei = wei.masked_fill(tril == 0, float('-inf'))
wei = F.softmax(wei, dim=-1)
xbow3 = wei @ x
torch.allclose(xbow, xbow3,atol=1e-6)

True

## 8. 单头自注意力：Q/K/V 登场

玩具例子里权重是**固定且均匀**的；真正的自注意力让权重**由数据决定**：

- **query（我在找什么）/ key（我有什么）/ value（我实际携带的信息）**：对 x 做的三个无偏置线性变换；
- `wei = q @ k.T`：每个位置对每个位置的「相似度分数」`(B,T,T)`；
- `masked_fill(tril==0, -inf)`：因果掩码，不许看未来（语言模型必须从左到右）；
- `softmax` 归一化 → `out = wei @ v`：按注意力权重聚合 value。

```
k = key(x); q = query(x); v = value(x)
wei = q @ k.transpose(-2,-1)          # (B,T,hs) @ (B,hs,T) → (B,T,T)
wei = wei.masked_fill(tril == 0, float('-inf'))
wei = F.softmax(wei, dim=-1)
out = wei @ v                          # (B,T,T) @ (B,T,hs) → (B,T,hs)
```

> 视频下一步会给 `q @ k.T` 补上 `1/√head_size` 缩放（防止点积过大、softmax 太尖锐）——这是 `v2.py` 与本 notebook 的差异之一。

In [19]:
torch.manual_seed(1337)
B,T,C = 4,8,32 # batch, time, channels
x = torch.randn(B,T,C)

# 让一个 Head 真正做一次自注意力（v2.py 里会被封装成 Head 类）
head_size = 16
# q/k/v 各做一次线性变换（C → head_size），角色不同：
# query = 我在找什么，key = 我有什么，value = 我实际携带的信息
key = nn.Linear(C, head_size, bias=False)
query = nn.Linear(C, head_size, bias=False)
value = nn.Linear(C, head_size, bias=False)
k = key(x)  # (B, T, 16)
q = query(x)  # (B, T, 16)
# 相似度分数：每个位置 × 每个位置 (B,T,T)；下一节会补 1/sqrt(head_size) 缩放
wei = q @  k.transpose(-2,-1)

tril = torch.tril(torch.ones(T, T))
# wei = torch.zeros((T,T))
wei = wei.masked_fill(tril == 0, float('-inf'))   # 因果掩码：右上角（未来）填 -inf
wei = F.softmax(wei, dim=-1)                      # 每行归一化成注意力权重

v=value(x)
out = wei @ v

out.shape

torch.Size([4, 8, 16])

In [ ]:
wei

## 9. 附：BatchNorm1d 小实验（与 Day8 对照）

从 makemore 搬来的遗留实验，顺手记一个关键区别：这里 `mean(1)` / `var(1)` 是沿**特征维**归一化（每个样本内部，LayerNorm 的思路），而不是 Day8 里沿 batch 维（dim=0）的 BatchNorm1d。GPT 里用的 `nn.LayerNorm` 正是这条路线。

（顺手修复：原文件里 `parameters()` 被错误缩进到 `__call__` 的 `return` 之后，是不可达的死代码，已挪回类方法位置。）

In [23]:
# 附：归一化小实验（从 makemore 搬来的遗留代码）
# 注意：这里沿 dim=1（特征维）归一化 = LayerNorm 思路；Day8 的 BatchNorm1d 是沿 dim=0（batch 维）
class BatchNorm1d:

  def __init__(self, dim, eps=1e-5, momentum=0.1):
    self.eps = eps
    # parameters (trained with backprop)
    self.gamma = torch.ones(dim)
    self.beta = torch.zeros(dim)

  def __call__(self, x):
    # calculate the forward pass
    xmean = x.mean(1, keepdim=True)  # 沿特征维求均值
    xvar = x.var(1, keepdim=True)    # 与 makemore 里沿 batch 维的版本对比着看
    xhat = (x-xmean) / torch.sqrt(xvar + self.eps)
    self.out = self.gamma * xhat + self.beta
    # update the buffers

    return self.out

  def parameters(self):
    return [self.gamma, self.beta]

torch.manual_seed(1337)
module = BatchNorm1d(100)
x = torch.randn(32, 100) # batch size 32 of 100-dimensional vectors
x = module(x)
x.shape

torch.Size([32, 100])

## 10. 进度：这里停在哪，下一步吃什么

本 notebook 停在**单头自注意力**（还没做缩放、多头、FFN、Block 堆叠）。

继续走完剩下的路看 **`v2.py`**（同文件夹）：它在同一个骨架上补齐了——

- 多头注意力（6 个 head 并行 + proj 合并回 n_embd）
- 前馈层 FFWD（4 倍扩宽 + ReLU + 投影回）
- 残差 + Pre-LN 的 Transformer Block × 6 层
- 权重初始化（0.02 正态）、完整训练循环、生成采样

对照阅读顺序：先用本 notebook 的 1–8 节理解每个零件，再读 `v2.py` 看它们如何组装成一个 ≈10.8M 参数的 GPT。